# Lesson 4.2 — Measuring the Gap

outcome_gap scores the predicted harvest against the actual one: which fruit diverged, in which direction, and a match flag.

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, model_perception, understand
from modules.module10.twin import DigitalTwin, GroundTruth, outcome_gap, snapshot
def victim_and_xy(seed=1):
    w = GreenhouseWorld.demo_row(n=6, seed=seed)
    dets = model_perception(w, rng=np.random.default_rng(7))
    v = understand(dets, w)[1]['id']
    vxy = next(d['xy'] for d in dets if d['id']==v)
    return v, vxy
checks = []
v, vxy = victim_and_xy()
ground = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1), unmodeled={v: dict(obstacle=(vxy,0.25))})
twin = DigitalTwin(ground.world)
gap = outcome_gap(twin.simulate(), ground.run())
print('gap -> n_diffs=%d match=%s' % (gap['n_outcome_diffs'], gap['match']))
print('  harvested_only_in_sim=%s  skipped_only_in_real=%s' % (gap['harvested_only_in_sim'], gap['skipped_only_in_real']))
checks.append(v in gap['harvested_only_in_sim'])    # twin too optimistic about v
checks.append(v in gap['skipped_only_in_real'])
checks.append(gap['n_outcome_diffs'] > 0 and gap['match'] is False)
# a no-effect case matches
g2 = GroundTruth(GreenhouseWorld.demo_row(n=6, seed=1))
t2 = DigitalTwin(g2.world)
clean_gap = outcome_gap(t2.simulate(), g2.run())
print('no unmodeled effect -> match:', clean_gap['match'])
checks.append(clean_gap['match'] is True)
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


gap -> n_diffs=2 match=False
  harvested_only_in_sim=['F3']  skipped_only_in_real=['F3']


no unmodeled effect -> match: True
4/4 checks passed.
All checks passed.
